# P05 - Basic RAG

## RAG 사전단계 (저장)
vectorstore 2개를 생성하여 각각 2개의 PDF를 load-split-embed-store
1. load -> PyPDF Loader 사용 -> (uv add pypdf langchain-community)
1. split -> 적절한 split 기준을 찾아서 적용 (우선은 `RecursiveCharacterTextSplitter` 사용)
1. embed -> OpenAI `text-embedding-3-small` 사용
1. store -> `InMemoryVectorStore` 사용하되, 총 2개의 vectorstore 생성해야함. 변수명은
    - `nvda_vectorstore`
    - `googl_vectorstore`

## Agent 구현단계
1개의 Agent에 2(3)개의 Tool 주기
1. `nvda_vectorstore` 를 감싸고 있는 Tool
1. `googl_vectorstore` 를 감싸고 있는 Tool
1. (Optional) `TavilySearchTool` 제공 가능
1. System Prompt 와 Tool Description 을 잘 작성하여 필요한 경우 필요한 Tool 호출하도록 제작

In [27]:
import os
from dotenv import load_dotenv

# 0. 환경 변수 로드 (.env에 OPENAI_API_KEY 설정 필요)
load_dotenv()


True

In [28]:

# ==========================================
# [Step 1] RAG 사전 단계: Load -> Split -> Embed -> Store
# ==========================================
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore

# 1. 문서 로드 (PDF 경로를 실제 파일 위치로 지정해주세요)
nvda_pdf_path = "data/NVDA-10-K-2025.pdf"   # 엔비디아 관련 PDF
googl_pdf_path = "data/GOOG-10-K-2025.pdf" # 구글 관련 PDF

nvda_loader = PyPDFLoader(nvda_pdf_path)
googl_loader = PyPDFLoader(googl_pdf_path)

nvda_docs = nvda_loader.load()
googl_docs = googl_loader.load()

# 2. 문서 분할 (Split)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    add_start_index=True, # 쪼개진 chunk의 시작
    separators=["\n\n", "\n", ".", " ", ""]
)

nvda_splits = text_splitter.split_documents(nvda_docs)
googl_splits = text_splitter.split_documents(googl_docs)

# 3. 임베딩 모델 정의 (text-embedding-3-small)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 4. 벡터 스토어 생성 (InMemoryVectorStore)
nvda_vectorstore = InMemoryVectorStore.from_documents(
    documents=nvda_splits,
    embedding=embeddings
)

googl_vectorstore = InMemoryVectorStore.from_documents(
    documents=googl_splits,
    embedding=embeddings
)

# 리트리버 설정 (관련성 높은 상위 3개 청크 검색)
nvda_retriever = nvda_vectorstore.as_retriever(search_kwargs={"k": 3})
googl_retriever = googl_vectorstore.as_retriever(search_kwargs={"k": 3})




In [33]:
# ==========================================
# [Step 2] Tool 정의 (함수 기반 @tool)
# ==========================================
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

@tool
def retrieve_nvidia_info(query: str) -> str:
    """NVIDIA(NVDA)의 실적, 공시, 사업 개요, 재무 상태 등 엔비디아 관련 질문에 대한 공식 문서 정보를 검색할 때 사용합니다."""
    docs = nvda_retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

@tool
def retrieve_google_info(query: str) -> str:
    """Google / Alphabet(GOOGL)의 실적, 사업 부문, 최신 기술, 공시 등 구글 관련 질문에 대한 공식 문서 정보를 검색할 때 사용합니다."""
    docs = googl_retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

tavily_search_tool = TavilySearch(max_results=5, topic="news")

tools = [retrieve_nvidia_info, retrieve_google_info, tavily_search_tool]


# (선택 사항) Tavily 웹 검색 툴 추가 시:
# if os.getenv("TAVILY_API_KEY"):
#    from langchain_tavily import TavilySearch
#    tavily_search_tool = TavilySearch(max_results=5, topic="news")



In [34]:
# ==========================================
# [Step 3] 최신 ReAct Agent 구현 및 실행
# ==========================================
# 최신 LangChain 표준인 langgraph 기반 create_react_agent 사용
# (langgraph가 설치되어 있지 않다면 LangChain 내장 AgentExecutor 방식 사용 가능)
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# LLM 정의
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# System Prompt 작성
system_prompt = (
    "You are an expert analyst specialized in answering questions about NVIDIA and Google (Alphabet).\n\n"
    "CRITICAL CONSTRAINTS & TOOL USAGE INSTRUCTIONS (MUST COMPLY):\n"
    "1. NEVER answer from your pretrained knowledge alone. You must always retrieve facts using the provided tools.\n"
    "2. TOOL SELECTION RULES:\n"
    "   - For NVIDIA's official reports, financial results, or business overviews: Call `retrieve_nvidia_info` first.\n"
    "   - For Google's official reports, financial results, or business overviews: Call `retrieve_google_info` first.\n"
    "   - For comparison across both companies: Call BOTH `retrieve_nvidia_info` and `retrieve_google_info`.\n"
    "   - For latest news, recent market events, industry trends, or if the internal PDF documents do not contain the answer: Use `tavily_search_tool` (or the registered web search tool) to fetch up-to-date web information.\n"
    "3. Base your answers strictly on the retrieved results (PDF contexts and web search results). Mention data sources clearly if applicable.\n"
    "4. If information is not found in either the PDF documents or the web search, state that clearly rather than assuming facts.\n"
    "5. LANGUAGE REQUIREMENT: All final answers must be formulated and written in fluent, professional Korean."
)

# Agent 생성
agent_executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt  # state_modifier 대신 prompt 사용
)


In [35]:
# ==========================================
# [Step 4] 테스트 실행
# ==========================================
if __name__ == "__main__":
    queries = [
        "엔비디아의 주요 매출원과 최근 분기 실적 요약해줘.",
        "구글과 엔비디아의 매출규모를 엔비디아 기준(100)으로 비교하고, 각각의 AI 관련 사업 방향을 문서 내용 기반으로 비교분석해줘."
    ]

    # for q in queries:
    #     print(f"\n[질문]: {q}")
    #     response = agent_executor.invoke({"messages": [HumanMessage(content=q)]})
    #     # 에이전트의 마지막 응답 메시지 출력
    #     final_message = response["messages"][-1].content
    #     print(f"[답변]:\n{final_message}\n" + "="*50)
        
for q in queries:
    print(f"\n[질문]: {q}")
    result = agent_executor.invoke({
        "messages": [
            {"role": "user", "content": q}
        ]
    })
    print(f"[답변]:\n{result['messages'][-1].content}\n" + "=" * 50)


[질문]: 엔비디아의 주요 매출원과 최근 분기 실적 요약해줘.
[답변]:
엔비디아의 주요 매출원과 최근 분기 실적에 대한 요약은 다음과 같습니다.

### 주요 매출원
엔비디아의 매출은 주로 두 가지 주요 부문에서 발생합니다:
1. **컴퓨팅 및 네트워킹 (Compute & Networking)**: 이 부문은 2026년 1월 25일 기준으로 193,479백만 달러의 매출을 기록하였으며, 전년 대비 67% 증가했습니다. 이 부문은 가속 컴퓨팅 및 인공지능(AI) 플랫폼의 주요 변화에 의해 주도되었습니다.
2. **그래픽스 (Graphics)**: 이 부문은 22,459백만 달러의 매출을 기록하였으며, 전년 대비 57% 증가했습니다.

### 최근 분기 실적
2026년 1월 25일 기준으로 엔비디아의 총 매출은 215,938백만 달러로, 전년 동기 대비 65% 증가했습니다. 운영 수익은 139,297백만 달러로, 전년 대비 58% 증가했습니다. 순이익은 55.6%로 보고되었습니다.

이러한 실적은 엔비디아가 AI 및 데이터 센터 시장에서의 강력한 수요에 힘입어 이루어진 것으로 보입니다.

[질문]: 구글과 엔비디아의 매출규모를 엔비디아 기준(100)으로 비교하고, 각각의 AI 관련 사업 방향을 문서 내용 기반으로 비교분석해줘.
[답변]:
### 매출 규모 비교

엔비디아와 구글의 매출 규모를 엔비디아 기준(100)으로 비교하면 다음과 같습니다.

- **엔비디아**: 215,938 백만 달러 (2025 회계연도)
- **구글**: 84,026 백만 달러 (2025 회계연도)

엔비디아의 매출을 100으로 설정할 경우, 구글의 매출은 약 38.9이 됩니다. 즉, 엔비디아의 매출이 구글의 매출보다 약 2.57배 더 큽니다.

### AI 관련 사업 방향 비교

#### 엔비디아
엔비디아는 AI 기술을 활용하여 다양한 산업에 걸쳐 가속화된 컴퓨팅 플랫폼을 제공하고 있습니다. 이들은 주로 다음과 같은 분야에서 사용됩니다:
- **제너레이티브 AI**: 새로운 제품 및